# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step procedures for loading, extracting, and exploring data from the [FAIR^2 package](https://sen.science/doi/10.71728/senscience.vq4a-28xa), which is defined using the Croissant schema and accessible with the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Title: ", getattr(metadata, 'name', None))
print("Description: ", getattr(metadata, 'description', None))

## 2. Data Overview
Review available record sets, their fields, and their `@id` identifiers.

Below we list the available record sets, the fields they contain, and their `@id` values.

In [ ]:
# List all available record sets and their field IDs
record_sets = getattr(metadata, 'record_set', getattr(metadata, 'recordSet', []))

if not record_sets:
    print("No record sets found in schema (possibly the Croissant uses 'recordSet' as ",
          "opposed to 'record_set'). Trying direct file inspection...")
    # As per schema, may need to access fields another way. Try get all 'RecordSet' entities from the metadata graph.
    from mlcroissant._src.structure_graph import StructureGraph
    sg = StructureGraph(croissant_url)
    all_recordsets = [obj for obj in sg.objects.values() if obj.type_.endswith('RecordSet')]
    if not all_recordsets:
        print("No record sets available in the dataset.")
    else:
        print(f"Found {len(all_recordsets)} record sets:")
        for rs in all_recordsets:
            print(f"  RecordSet @id: {rs.id}")
            # Print field ids for this record set
            field_ids = getattr(rs, 'field', getattr(rs, 'fields', []))
            print(f"    Fields (@id): {field_ids if field_ids else 'Not listed'}")
    record_set_ids = [rs.id for rs in all_recordsets]
else:
    record_set_ids = [getattr(rs, '@id', getattr(rs, 'id', rs)) for rs in record_sets]
    print(f"Found {len(record_set_ids)} record sets:")
    for rs in record_sets:
        print(f"  RecordSet @id: {getattr(rs, '@id', getattr(rs, 'id', rs))}")
        fields = getattr(rs, 'field', getattr(rs, 'fields', []))
        print(f"    Fields (@id): {fields if fields else 'Not listed'}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

If the dataset contains multiple record sets, we load the first one as an example, but you can change `selected_record_set_id` to any valid `@id` listed above. For full exploration, all record sets are loaded into pandas DataFrames.

In [ ]:
# The previous cell should have collected all valid record set @ids as record_set_ids
if 'record_set_ids' not in locals():
    # Fallback: Try from metadata again
    record_sets = getattr(metadata, 'record_set', getattr(metadata, 'recordSet', []))
    record_set_ids = [getattr(rs, '@id', getattr(rs, 'id', rs)) for rs in record_sets]
# Pick the first record set as example, but feel free to choose another one
if not record_set_ids:
    raise ValueError("No record sets available in this dataset.")

selected_record_set_id = record_set_ids[0]
print(f"Using record set @id: {selected_record_set_id}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print("  Columns:", df.columns.tolist())
        print(df.head(2))
    except Exception as e:
        print(f"  Failed to load records for {record_set_id}: {e}")

# Now preview the columns of selected_record_set_id
if selected_record_set_id in dataframes:
    print('Columns:', dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print(f"No data loaded for {selected_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter records based on a numeric field, normalize, and group by a categorical field.

Select a numeric field and a group field from your record set. If you are unsure, check the DataFrame columns output above and modify the variables below as needed. All fields are referenced strictly by their `@id`.

In [ ]:
# --- Configuration ---
df = dataframes[selected_record_set_id]
print("Available columns:", df.columns.tolist())

# Guess (or update below if necessary) - select a numeric column
numeric_field_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]

numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]  # Adjust as necessary
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Or you can manually choose e.g.:
# numeric_field_id = '@your-numeric-field-id'

# Filtering
THRESHOLD = df[numeric_field_id].mean() if numeric_field_id in df else 0
filtered_df = df[df[numeric_field_id] > THRESHOLD]
print(f"Filtered records with {numeric_field_id} > {THRESHOLD:.3f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group (categorical) field
group_field_candidates = [col for col in df.columns if df[col].dtype == 'object']
group_field_id = group_field_candidates[0] if group_field_candidates else df.columns[0]
print(f"Selected group field: {group_field_id}")

# Group and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id} (mean of numeric fields):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions for selected fields in the dataset. Update the field names below as appropriate or use those determined above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric variable
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot of numeric field by group_field
if group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(y=df[numeric_field_id], x=df[group_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=60)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, you have learned how to:
- Programmatically load a Croissant dataset package with `mlcroissant`.
- Explore available record sets, their fields, and `@id`s.
- Load and manipulate the datasets in pandas DataFrames.
- Perform simple exploratory data analysis (filtering, normalization, and grouping) on specified numeric fields with references by `@id`.
- Visualize data using matplotlib and seaborn.

Further analysis can be performed depending on your particular research or application needs. For full reproducibility, always track the `@id`s of all entities and reference them in your code and documentation. For more information on the dataset structure and semantics, see the [Croissant specification](https://mlcommons.org/croissant/spec/).